# Canonical orientation of Visium sections

Goal: rotate each sample so all coronal sections face the same way in figure panels. Iterate by eye on this notebook, then write an oriented combined h5ad that downstream notebooks read.

Workflow
1. Load combined h5ad.
2. Render current orientation grid.
3. Edit the `rotations` dict in the cell below until grid looks aligned.
4. Save dict back to `data/rotations.yaml`.
5. Write oriented h5ad to `ST_BRICHOS_oriented.h5ad`.


In [ ]:
from __future__ import annotations

import math
import sys
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from utils.canonical_orientation import (
    rotate_visium_sample, apply_orientation_table
)

BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_IN  = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD_OUT = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
ROT_YAML = ROOT / 'data' / 'rotations.yaml'

SAMPLE_KEY    = 'library_id'   # change to 'sample_id' if needed
TREATMENT_KEY = 'treatment'
TREAT_COLOR = {'WT': '#4a4a4a', 'PBS': '#1f77b4', 'BRICHOS': '#d62728'}

print('project root:', ROOT)
print('input h5ad :', H5AD_IN)


## 1. Load combined h5ad

Reload by re-running this cell. Cheaper than re-running everything.

In [ ]:
adata = sc.read_h5ad(H5AD_IN)
print(adata)
if SAMPLE_KEY not in adata.obs.columns:
    candidates = [c for c in adata.obs.columns
                  if 'sample' in c.lower() or 'library' in c.lower()]
    print(f'!! {SAMPLE_KEY!r} not in obs. candidates:', candidates)
libs = sorted(adata.uns['spatial'].keys())
print(f'{len(libs)} libraries:', libs)


## 2. Helper: render orientation grid

Crosshair marks image centre — handy for judging tilt against the midline. Spot dots coloured by treatment.

In [ ]:
def render_grid(adata, rotations, title, ncols=4, figscale=2.8):
    libs = sorted(adata.uns['spatial'].keys())
    n = len(libs)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figscale * ncols, (figscale + 0.2) * nrows),
                             squeeze=False)
    for ax, lib in zip(axes.flat, libs):
        sp = adata.uns['spatial'][lib]
        img = sp['images']['hires']
        sf = sp['scalefactors']['tissue_hires_scalef']
        ax.imshow(img, origin='upper')

        mask = (adata.obs[SAMPLE_KEY] == lib).to_numpy()
        xy = adata.obsm['spatial'][mask] * sf
        treat = (adata.obs.loc[mask, TREATMENT_KEY].iloc[0]
                 if mask.any() and TREATMENT_KEY in adata.obs.columns
                 else '?')
        col = TREAT_COLOR.get(treat, '#888')
        ax.scatter(xy[:, 0], xy[:, 1], s=0.4, c=col,
                   alpha=0.55, edgecolors='none')

        H, W = img.shape[:2]
        ax.axhline(H / 2, color='#888', linewidth=0.4, alpha=0.5)
        ax.axvline(W / 2, color='#888', linewidth=0.4, alpha=0.5)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)

        spec = rotations.get(lib, {})
        a = spec.get('angle_deg', 0)
        f = spec.get('flip_h', False)
        suffix = f' / rot {a:+g}' + ('/flip' if f else '')
        ax.set_title(f'{lib}  [{treat}]{suffix}',
                     fontsize=8, color=col, loc='left', pad=2)

    for ax in axes.flat[n:]:
        ax.axis('off')

    fig.suptitle(title, fontsize=10, y=0.995)
    fig.tight_layout()
    return fig


## 3. Current orientation (no rotation applied)

In [ ]:
_ = render_grid(adata, {}, 'Current orientation')
plt.show()


## 4. Edit rotations

* `angle_deg` — counter-clockwise in the displayed image.
* `flip_h`    — mirror left/right *after* rotation.

Initial values loaded from `data/rotations.yaml`. Edit, re-run the preview cell below, repeat. When happy, run the save cell.

In [ ]:
if ROT_YAML.exists():
    with open(ROT_YAML) as f:
        rotations = yaml.safe_load(f) or {}
else:
    rotations = {lib: {'angle_deg': 0, 'flip_h': False} for lib in libs}

# ensure every library has an entry
for lib in libs:
    rotations.setdefault(lib, {'angle_deg': 0, 'flip_h': False})

# --- edit angles / flips here -----------------------------------------
# rotations['P24215_101']['angle_deg'] = 15
# rotations['P28052_201']['flip_h']   = True
# ---------------------------------------------------------------------

rotations


## 5. Preview with rotations applied

Works on a deep copy so the original `adata` stays clean. Re-run after editing the dict above.

In [ ]:
adata_rot = adata.copy()
# images/uns aren't deep-copied by AnnData.copy(); fix that
adata_rot.uns = deepcopy(adata.uns)
apply_orientation_table(adata_rot, rotations,
                        sample_key=SAMPLE_KEY, verbose=True)
_ = render_grid(adata_rot, rotations, 'After rotations.yaml')
plt.show()


## 6. Save rotations.yaml + write oriented h5ad

Run when the previous preview looks aligned. Writes:
* `data/rotations.yaml` (so the script tools see the same values)
* `notebooks/data/ST_BRICHOS_oriented.h5ad` (downstream notebooks read this)

In [ ]:
ROT_YAML.parent.mkdir(parents=True, exist_ok=True)
with open(ROT_YAML, 'w') as f:
    yaml.safe_dump(rotations, f, sort_keys=False)
print('wrote', ROT_YAML)

print('writing', H5AD_OUT, '...')
adata_rot.write_h5ad(H5AD_OUT, compression='gzip')
print('done.')


## 7. Sanity check — read oriented file back

In [ ]:
adata_check = sc.read_h5ad(H5AD_OUT)
_ = render_grid(adata_check, rotations,
                'Sanity check — reloaded oriented h5ad')
plt.show()
